# RoadShield Vision — train the road-damage detector on RDD2022 (GPU)

Runs the same reproducible pipeline as `ml/` in the repository:
1. download RDD2022 **from its official Figshare record** with licence + MD5 verification,
2. prepare (VOC → JSON/YOLO, duplicate grouping, per-country 80/10/10 split, versioned manifest),
3. fine-tune torchvision Faster R-CNN MobileNetV3-FPN (BSD-3-Clause), 4. evaluate on the held-out test split,
5. export ONNX + `metrics.json` and register the result as a **candidate** (deployment needs approval).

Runtime → Change runtime type → **T4 GPU**. Full RDD2022 is ~12 GB; select countries with `--match` to save time.

Do not report metrics from a smoke test (`--max-images`) as model accuracy.

In [ ]:
!nvidia-smi -L || echo 'No GPU — training will be very slow'
!git clone --depth 1 https://github.com/Alaa2134/roadshield-ai.git
%cd roadshield-ai
!pip -q install torch torchvision pillow numpy onnx onnxscript

## 1. Download (official source, licence-checked)

In [ ]:
!python ml/datasets/download_rdd2022.py --out /content/rdd2022 --dry-run
!cat /content/rdd2022/LICENSE_RECORD.json | head -40

In [ ]:
# Download (all files, or pass --match with a file-name fragment shown above)
!python ml/datasets/download_rdd2022.py --out /content/rdd2022
!cd /content/rdd2022 && for z in *.zip; do unzip -q -o "$z"; done; ls /content/rdd2022

## 2. Prepare

In [ ]:
RAW = '/content/rdd2022/RDD2022'  # adjust if the archive extracts to a different folder
!python ml/datasets/prepare_rdd2022.py --raw {RAW} --out /content/rdd2022_prepared --seed 42
!cat /content/rdd2022_prepared/MANIFEST.json | head -60

## 3–4. Train and evaluate

In [ ]:
!python ml/detector/train_detector.py --data /content/rdd2022_prepared --raw {RAW} \
    --out /content/runs/rs-frcnn-mbv3 --epochs 24 --batch 8 --pretrained --onnx

In [ ]:
import json; m = json.load(open('/content/runs/rs-frcnn-mbv3/metrics.json'))
print(json.dumps({k: m['test'][k] for k in ('mAP50','mAP50-95','precision','recall','f1')}, indent=2))
print(json.dumps(m['per_country'], indent=2)); print(m['latency_ms'])

## 5. Save artefacts

Download `model.onnx`, `model.pt`, `metrics.json` and `history.json`, then copy `metrics.json` into `docs/test-results/` and update `docs/TRAINING_REPORT.md` / `docs/MODEL_CARD.md`. Register the candidate in the AI Model Center (platform admin) — it can only be deployed after review.

In [ ]:
from google.colab import files
for f in ('metrics.json','history.json','model.onnx'):
    files.download(f'/content/runs/rs-frcnn-mbv3/{f}')